In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.utils import resample
import numpy as np
import matplotlib as mpl
import pandas as pd

#----for evi -----

df1 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/UMedPT_LR/combine_origin.csv")
df2 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/UMedPT_LR/axial_original.csv")
df3 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/UMedPT_LR/sagittal_original.csv")
df4 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/FMmodel192/test/best_FM_192.csv")

# Extract ground truth and predicted probabilities for EVI and MFI
y_true = df1['y_test_evi'].values

y_pred_model1 = df1['y_pred_prob_evi'].values
y_pred_model2 = df2['y_pred_prob_evi'].values
y_pred_model3 = df3['y_pred_prob_evi'].values
y_pred_model4 = df4['predicted_probability_evi'].values

# Model names and placeholder for AUC formatting
models = {
    'UMedPT_LR axial+sagittal': y_pred_model1,
    'UMedPT_LR axial': y_pred_model2,
    'UMedPT_LR sagittal': y_pred_model3,
    'UMedPT axial': y_pred_model4,
}

# Soft, elegant colors: light blue, light yellow, light green, light red
# colors = ['#d62728','#1f77b4', '#ffbb00', '#2ca02c'] # soft  red blue, yellow, green
# colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
# colors = ['#FBDD85', '#80A6E2', '#F46F43', '#403990']
# colors = ['#403990', '#F46F43','#80A6E2','#FBDD85']
colors = ['#403990', '#F46F43','#80A6E2','#FBDD85']

# Set plot style to match high-quality publication standards (e.g., Nature)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.size'] = 10

#bootstrap compute CI
def bootstrap_roc_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.95):
    rng = np.random.RandomState(42)
    tprs = []
    base_fpr = np.linspace(0, 1, 101)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(y_true[indices])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true[indices], y_pred[indices])
        tpr_interp = np.interp(base_fpr, fpr, tpr)
        tpr_interp[0] = 0.0
        tprs.append(tpr_interp)

    tprs = np.array(tprs)
    mean_tpr = tprs.mean(axis=0)
    std_tpr = tprs.std(axis=0)
    tpr_upper = np.minimum(mean_tpr + 1.96 * std_tpr, 1)
    tpr_lower = np.maximum(mean_tpr - 1.96 * std_tpr, 0)
    return base_fpr, mean_tpr, tpr_lower, tpr_upper

def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000, seed=42, alpha=0.95):
    rng = np.random.RandomState(seed)
    stats = []
    n = len(y_pred)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        stats.append(roc_auc_score(y_true[idx], y_pred[idx]))
    lower = np.percentile(stats, (1 - alpha) / 2 * 100)
    upper = np.percentile(stats, (1 + alpha) / 2 * 100)
    return lower, upper


# Create the figure
plt.figure(figsize=(6, 6))

# Plot ROC curves for each model
# for (label, y_pred), color in zip(models.items(), colors):
#     fpr, mean_tpr, lower_tpr, upper_tpr = bootstrap_roc_ci(y_true, y_pred)
#     auc_score = auc(fpr, mean_tpr)
#     plt.plot(fpr, mean_tpr, lw=2, label=f"{label} (AUC: {auc_score:.2f})", color=color)
#     # plt.plot(fpr, lower_tpr, color=color, linestyle='--', alpha=0.6, linewidth=1)
#     # plt.plot(fpr, upper_tpr, color=color, linestyle='--', alpha=0.6, linewidth=1)
#     plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.12)

shade_first_n = 2  
for i, ((model_name, y_pred), color) in enumerate(zip(models.items(), colors)):
    fpr_raw, tpr_raw, _ = roc_curve(y_true, y_pred)
    auc_raw = roc_auc_score(y_true, y_pred) 

     # --- AUC bootstrap CI ?? ---
    auc_lo, auc_hi = bootstrap_auc_ci(y_true, y_pred)

    # Legend ??????? or en-dash?
    label_text = f"{model_name} (AUC: {auc_raw:.2f} [{auc_lo:.2f}-{auc_hi:.2f}])"

    # ?????? or ?? plt.plot ???????
    plt.step(fpr_raw, tpr_raw, where='post', lw=2.2, color=color, label=label_text)

    # --- bootstrap CI for shading / dashed lines ---
    # fpr_ci, mean_tpr, lower_tpr, upper_tpr = bootstrap_roc_ci(y_true, y_pred)
    
    # if i < shade_first_n:
    #     plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.1, linewidth=0)
    # else:
    #     plt.plot(fpr, lower_tpr, color=color, linestyle='--', linewidth=1.1, alpha=0.7)
    #     plt.plot(fpr, upper_tpr, color=color, linestyle='--', linewidth=1.1, alpha=0.7)

# Plot diagonal line representing random guess
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess')

# Set titles and labels
plt.title('Best 4 Models for EVI Classification', fontsize=12, weight='normal')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)

plt.legend(loc='lower right', frameon=False, fontsize=9)
plt.grid(True, linestyle='--', alpha=0.3)
plt.xticks(np.linspace(0, 1, 6))
plt.yticks(np.linspace(0, 1, 6))

# Thicker border lines
for spine in plt.gca().spines.values():
    spine.set_linewidth(1.2)

output_dir = "/path/to/workspace/Complete_set/Experiments/Results/ROC PHOTO/"
filename = "roc_curve_evi_new3.png"

# Optimize layout
plt.tight_layout()
plt.savefig(output_dir + filename, dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib as mpl
import pandas as pd

#----for MFI-----

df1 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/axial_harmo_FM192/best_FM_axial_harmo.csv")
df2 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/UMedPT_LR/axial_original.csv")
df3 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/sagittal_harmo/best_scr_segresnet_sagittalharmo.csv")
df4 = pd.read_csv(r"/path/to/workspace/Complete_set/Experiments/Results/Segresnet/best_segresnet_axial_original.csv")

# Extract ground truth and predicted probabilities for EVI and MFI
y_true = df1['ground_truth_mfi'].values

y_pred_model1 = df1['predicted_probability_mfi'].values
y_pred_model2 = df2['y_pred_prob_mfi'].values
y_pred_model3 = df3['predicted_probability_mfi'].values
y_pred_model4 = df4['predicted_probability_mfi'].values


# Model names and placeholder for AUC formatting
models = {
    'UMedPT axial_harmo': y_pred_model1,
    'UMedPT_LR axial': y_pred_model2,
    'SeResnet axial': y_pred_model4,
    'SeResnet sagittal_harmo': y_pred_model3
}

# Soft, elegant colors: light blue, light yellow, light green, light red
# colors = ['#d62728','#1f77b4', '#ffbb00', '#2ca02c'] # soft  red blue, yellow, green
colors = ['#403990', '#F46F43','#80A6E2','#FBDD85']

# Set plot style to match high-quality publication standards (e.g., Nature)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.size'] = 10

def bootstrap_roc_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.95):
    rng = np.random.RandomState(42)
    tprs = []
    base_fpr = np.linspace(0, 1, 101)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(y_true[indices])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true[indices], y_pred[indices])
        tpr_interp = np.interp(base_fpr, fpr, tpr)
        tpr_interp[0] = 0.0
        tprs.append(tpr_interp)

    tprs = np.array(tprs)
    mean_tpr = tprs.mean(axis=0)
    std_tpr = tprs.std(axis=0)
    tpr_upper = np.minimum(mean_tpr + 1.96 * std_tpr, 1)
    tpr_lower = np.maximum(mean_tpr - 1.96 * std_tpr, 0)
    return base_fpr, mean_tpr, tpr_lower, tpr_upper

def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000, seed=42, alpha=0.95):
    rng = np.random.RandomState(seed)
    stats = []
    n = len(y_pred)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        stats.append(roc_auc_score(y_true[idx], y_pred[idx]))
    lower = np.percentile(stats, (1 - alpha) / 2 * 100)
    upper = np.percentile(stats, (1 + alpha) / 2 * 100)
    return lower, upper

# Create the figure
plt.figure(figsize=(6, 6))

shade_first_n = 2  
for i, ((model_name, y_pred), color) in enumerate(zip(models.items(), colors)):
    fpr_raw, tpr_raw, _ = roc_curve(y_true, y_pred)
    auc_raw = roc_auc_score(y_true, y_pred) 

     # --- AUC bootstrap CI ?? ---
    auc_lo, auc_hi = bootstrap_auc_ci(y_true, y_pred)

    # Legend ??????? or en-dash?
    label_text = f"{model_name} (AUC: {auc_raw:.2f} [{auc_lo:.2f}-{auc_hi:.2f}])"

    # ?????? or ?? plt.plot ???????
    plt.step(fpr_raw, tpr_raw, where='post', lw=2.2, color=color, label=label_text)

    # # --- bootstrap CI for shading / dashed lines ---
    # fpr_ci, mean_tpr, lower_tpr, upper_tpr = bootstrap_roc_ci(y_true, y_pred)
    
    # if i < shade_first_n:
    #     plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.1, linewidth=0)
    # else:
    #     plt.plot(fpr, lower_tpr, color=color, linestyle='--', linewidth=1.1, alpha=0.7)
    #     plt.plot(fpr, upper_tpr, color=color, linestyle='--', linewidth=1.1, alpha=0.7)

# Plot diagonal line representing random guess
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess')

# Set titles and labels
plt.title('Best 4 Models for MFI Classification', fontsize=12, weight='normal')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)

# Add legend in bottom right, no frame
plt.legend(loc='lower right', frameon=False, fontsize=9)

# Add light grid lines
plt.grid(True, linestyle='--', alpha=0.3)

# Customize ticks
plt.xticks(np.linspace(0, 1, 6))
plt.yticks(np.linspace(0, 1, 6))

# Thicker border lines
for spine in plt.gca().spines.values():
    spine.set_linewidth(1.2)

output_dir = "/path/to/workspace/Complete_set/Experiments/Results/ROC PHOTO/"
filename = "roc_curve_mfi_new3.png"

# Optimize layout
plt.tight_layout()
plt.savefig(output_dir + filename, dpi=300, bbox_inches='tight')
plt.show()
